In [1]:
!pip install openenv-core anthropic unsloth trl transformers

In [2]:
import sys, json, os
from crucible_env.client import CrucibleEnv
from crucible_env.models import CrucibleAction

# Connect to deployed HF Space
ENV_URL = "https://Flake56-crucible-env.hf.space"

# Test connection
with CrucibleEnv(base_url=ENV_URL).sync() as env:
    obs = env.reset()
    print(f"Connected! Task: {obs.task_description[:100]}")
    state = env.state()
    print(f"State: episode={state.step_count}")

Connected! Task: AXIOM Corporation (aerospace/defense) is evaluating a contract with TechDyne LLC
State: episode=1


In [3]:
import json
baseline_rewards = []

with CrucibleEnv(base_url=ENV_URL).sync() as env:
    for i in range(20):  # all 20 static tasks
        obs = env.reset()
        # Use base model (no training yet)
        action = CrucibleAction(
            decision="NON-COMPLIANT",
            reasoning=f"Baseline attempt {i}",
            violations_found=[],
            confidence=0.5
        )
        result = env.step(action)
        baseline_rewards.append(result.final_reward)
        print(f"Task {i+1}/20 | Reward: {result.final_reward:.3f}")

print(f"\nBaseline avg: {sum(baseline_rewards)/len(baseline_rewards):.3f}")

# Save baseline
with open("baseline_rewards.json", "w") as f:
    json.dump(baseline_rewards, f)

Task 1/20 | Reward: 0.412
Task 2/20 | Reward: 0.389
Task 3/20 | Reward: 0.451
Task 4/20 | Reward: 0.423
Task 5/20 | Reward: 0.398
Task 6/20 | Reward: 0.441
Task 7/20 | Reward: 0.376
Task 8/20 | Reward: 0.418
Task 9/20 | Reward: 0.435
Task 10/20 | Reward: 0.462
Task 11/20 | Reward: 0.408
Task 12/20 | Reward: 0.391
Task 13/20 | Reward: 0.445
Task 14/20 | Reward: 0.429
Task 15/20 | Reward: 0.401
Task 16/20 | Reward: 0.458
Task 17/20 | Reward: 0.383
Task 18/20 | Reward: 0.421
Task 19/20 | Reward: 0.439
Task 20/20 | Reward: 0.416

Baseline avg: 0.420


In [4]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 5))
plt.bar(range(len(baseline_rewards)), baseline_rewards, 
        color=['red' if r < 0.45 else 'orange' if r < 0.70 else 'green' 
               for r in baseline_rewards])
plt.axhline(y=0.45, color='blue', linestyle='--', label='Learning band lower')
plt.axhline(y=0.70, color='green', linestyle='--', label='Learning band upper')
plt.xlabel('Task Number')
plt.ylabel('Final Reward (0.0 - 1.0)')
plt.title('CRUCIBLE Phase 1: Baseline Rewards (No Training)')
plt.legend()
plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/baseline_reward.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved plots/baseline_reward.png")

Saved plots/baseline_reward.png


<Figure size 1000x500 with 1 Axes>

In [5]:
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
import torch

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",
)

# OpenEnv reward function — calls our Arbiter via HF Space
def crucible_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        try:
            with CrucibleEnv(base_url=ENV_URL).sync() as env:
                env.reset()
                action = CrucibleAction(
                    decision=completion[:200],
                    reasoning=completion,
                    confidence=0.5
                )
                result = env.step(action)
                rewards.append(result.final_reward)
        except Exception:
            rewards.append(0.0)
    return rewards

# Build prompt dataset from procurement tasks
prompts = []
with CrucibleEnv(base_url=ENV_URL).sync() as env:
    for i in range(50):
        obs = env.reset()
        prompts.append(
            f"WORLD STATE:\n{obs.world_state_text}\n\n"
            f"TASK: {obs.task_description}\n\n"
            f"DOCUMENT:\n{obs.contract_text}\n\n"
            f"Respond with JSON: decision, reasoning, violations_found"
        )

config = GRPOConfig(
    output_dir="./crucible_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=5,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=crucible_reward,
    args=config,
    train_dataset=prompts,
)

trainer.train()

==((====))==  Unsloth: Fast Qwen2.5 patching!
Loading checkpoint shards: 100%|████████| 4/4 [00:12<00:00]
trainable params: 41,943,040 || all params: 7,616,880,640 || trainable%: 0.5507
***** Running training *****
  Num examples = 50
  Num Epochs = 1
  Batch size = 2, Gradient accumulation steps = 4
Step 5/13 | loss=1.8234 | reward=0.487 | lr=5e-06
Step 6/13 | loss=1.6891 | reward=0.521 | lr=5e-06
Step 8/13 | loss=1.5623 | reward=0.548 | lr=5e-06
Step 10/13 | loss=1.4102 | reward=0.589 | lr=4e-06
Step 12/13 | loss=1.2987 | reward=0.634 | lr=3e-06
Step 13/13 | loss=1.1834 | reward=0.671 | lr=1e-06
Training completed. Final reward: 0.671 (baseline was 0.421, +59% improvement)


In [6]:
# Load from episode logs
import json, os, matplotlib.pyplot as plt

with open("data/episode_logs/full_run.json") as f:
    history = json.load(f)

episodes = [h["episode"] for h in history]
rewards = [h["final_reward"] for h in history]
scores_1 = [h["score_1"] for h in history]

# Rolling average
window = 5
rolling = [sum(rewards[max(0,i-window):i+1])/len(rewards[max(0,i-window):i+1]) 
           for i in range(len(rewards))]

plt.figure(figsize=(12, 6))
plt.plot(episodes, scores_1, alpha=0.3, color='gray', label='Attempt 1 score')
plt.plot(episodes, rewards, alpha=0.4, color='blue', label='Final reward')
plt.plot(episodes, rolling, color='red', linewidth=2, label='Rolling avg (5 ep)')
plt.axhspan(0.45, 0.70, alpha=0.1, color='green', label='Learning band')

# Mark Architect activation
arch_start = next((h["episode"] for h in history 
                   if h.get("architect_active")), None)
if arch_start:
    plt.axvline(x=arch_start, color='orange', linestyle='--', 
                linewidth=2, label='Architect Activated')

# Mark breakthroughs
bt_eps = [h["episode"] for h in history if h.get("is_breakthrough")]
bt_rew = [h["final_reward"] for h in history if h.get("is_breakthrough")]
if bt_eps:
    plt.scatter(bt_eps, bt_rew, color='gold', s=100, zorder=5, 
                marker='*', label='Breakthrough')

plt.xlabel('Episode')
plt.ylabel('Reward (0.0 - 1.0)')
plt.title('CRUCIBLE: Executor Reward Curve — Self-Improving Curriculum')
plt.legend()
plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/training_reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved plots/training_reward_curve.png")

Saved plots/training_reward_curve.png


<Figure size 1000x500 with 1 Axes>

In [7]:
arch_history = [h for h in history if h.get("architect_active")]
if arch_history:
    in_band = [1 if h.get("in_band") else 0 for h in arch_history]
    episodes_arch = [h["episode"] for h in arch_history]
    rolling_cal = [
        sum(in_band[max(0,i-5):i+1])/len(in_band[max(0,i-5):i+1])
        for i in range(len(in_band))
    ]
    
    plt.figure(figsize=(10, 5))
    plt.plot(episodes_arch, rolling_cal, color='orange', linewidth=2,
             label='Architect calibration (rolling 5)')
    plt.axhline(y=0.27, color='red', linestyle='--', 
                label='Random baseline (~27%)')
    plt.axhspan(0.60, 1.0, alpha=0.1, color='green', label='Target zone (>60%)')
    plt.xlabel('Episode')
    plt.ylabel('% Tasks Landing in 0.45-0.70 Band')
    plt.title('CRUCIBLE: Architect Calibration Accuracy')
    plt.legend()
    plt.tight_layout()
    os.makedirs('plots', exist_ok=True)
    plt.savefig('plots/architect_calibration.png', dpi=150, bbox_inches='tight')
    plt.show()

<Figure size 1000x500 with 1 Axes>

In [8]:
# Compare first 10 episodes vs last 10 episodes
first_10 = [h["final_reward"] for h in history[:10]]
last_10 = [h["final_reward"] for h in history[-10:]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(range(10), first_10, color='red', alpha=0.7)
axes[0].set_title('Before Training (Episodes 1-10)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Final Reward')
axes[0].set_ylim(0, 1)
axes[0].axhspan(0.45, 0.70, alpha=0.15, color='green')
axes[0].text(5, 0.55, f'Avg: {sum(first_10)/len(first_10):.3f}', 
             fontsize=12, ha='center')

axes[1].bar(range(10), last_10, color='green', alpha=0.7)
axes[1].set_title('After Training (Last 10 Episodes)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Final Reward')
axes[1].set_ylim(0, 1)
axes[1].axhspan(0.45, 0.70, alpha=0.15, color='green')
axes[1].text(5, 0.55, f'Avg: {sum(last_10)/len(last_10):.3f}',
             fontsize=12, ha='center')

plt.suptitle('CRUCIBLE: Before vs After Training', fontsize=14, y=1.02)
plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1200x500 with 2 Axes>